# 17. Stage Bottleneck and Interaction Diagnostics — Herbal Supplements

This notebook diagnoses where the target item is gained, lost, promoted, or left unchanged across the retrieve-then-rerank pipeline. It combines the Notebook 14 canonical rank layer with the frozen Notebook 09 candidate pools to report Stage 1 exposure transitions, candidate overlap, rank promotion, rescue and harm counts, and QCHS-fallback diagnostics.

The appended opportunity decomposition separates target absence, exposure rescue or loss, promotion into or out of the top five, and within-top-five movement. The effective-n and minimum-detectable-effect calculation is an approximate post-hoc power diagnostic based on the observed paired-difference standard deviation. It does not convert a non-significant result into evidence of no effect.

Two post-hoc interaction contrasts are also retained. Interaction I compares Full minus RankP with the raw Stage 1 S1-P minus S1-Q difference. Interaction II is the fitted-policy 2×2 contrast:

\[
(\mathrm{Full-RankP})-(\mathrm{RetrP-Base}).
\]

Interaction II is the interaction reported as Ī in the thesis. It is available for LightGBM, GAM, and Transformer. Shannon Entropy is omitted because it has no separately fitted RetrP arm. Both interactions use 2,000 case/user bootstrap replicates with seed 42 and remain descriptive; they have no randomization p-value or multiplicity-adjusted decision.

A confidence interval containing zero indicates that departure from additivity was not detected. It does not establish additivity, equivalence, or complementarity. The executable verdict strings that use `additive_complementarity` are therefore interpretive labels rather than statistical conclusions.

Three provenance limitations remain. First, the executable `NOTEBOOK_NAME` is `22_stage_bottleneck_decomposition_herbal.ipynb` rather than the repository’s `17_...` filename. Second, the Stage 2 model-contract diagnostic reports `missing_notebook14_model_contract_hashes` because it searches the Notebook 14 source-inventory representation rather than the source-contract gate carrying those hashes. Third, Interaction II retains legacy `Y10` variable and output-column names even though the active condition code is `b02`. These defects do not change the stored decompositions but must not be presented as clean provenance.

The received notebook contains 16 cells: three Markdown cells and 13 executed code cells. Seven code cells contain stored outputs, no stored error is present, and execution counts are sequential from 1 through 13.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ==== Imports ====
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json
import math

import numpy as np
import pandas as pd

In [3]:
# ==== Config: Direct Notebook 14 and Notebook 09 Contracts ====
NOTEBOOK_NAME = "22_stage_bottleneck_decomposition_herbal.ipynb"
CATEGORY_ID = "herbal"
CATEGORY_KEY = "herbal_supplements"
CATEGORY_LABEL = "Herbal Supplements"

PROJECT_ROOT = Path(f"/content/drive/MyDrive/thesis_recsys/categories/{CATEGORY_KEY}")
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
ANALYSIS_DIR = OUTPUTS_DIR / "analysis"
OUT_DIR = ANALYSIS_DIR / "stage_bottleneck_decomposition"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_AGGREGATE_DIR = OUTPUTS_DIR / "pipeline_aggregate"
PIPELINE_MANIFEST_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_manifest.json"
CANONICAL_RAW_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_canonical_per_case_metrics.parquet"
NB14_AVAILABILITY_PATH = PIPELINE_AGGREGATE_DIR / "pipeline_method_depth_availability.csv"
STAGE1_MANIFEST_PATH = (
    OUTPUTS_DIR / "stage1_candidate_pools"
    / f"stage1_candidate_pool_export_manifest_{CATEGORY_ID}.json"
)

# Canonical stage-allocation vocabulary as EMITTED by the executed Notebook 14
# (verified against 14_pipeline_aggregate_{face,herbal}_selfpool_2x2_REVISED outputs):
#   P0 = S1-Q, P1-only = S1-P, P2-Q = Base, P2-P = RankP, b02 = RetrP, Full = Full.
CANONICAL_CONDITIONS = ["P0", "P1-only", "P2-Q", "P2-P", "b02", "Full"]
# The 2x2 stage-2 cells. b02 is the retrieval-personalised / no-prior arm.
STAGE2_CONDITIONS = ["P2-Q", "P2-P", "b02", "Full"]
# The promotion ladder used for the family-completeness and GAM-depth contracts.
# b02 is deliberately excluded here: (shannon, b02) is design-excluded, so a
# ladder that included b02 could not assert completeness over all four families.
PROMOTION_LADDER_CONDITIONS = ["P2-Q", "P2-P", "Full"]
RERANKER_FAMILIES = ["shannon", "lightgbm", "gam", "transformer"]
PRIMARY_PROMOTION_CUTOFF = 5

OUTPUT_FILES = {
    "exposure_transition": OUT_DIR / "bottleneck_exposure_transition.csv",
    "exposure_by_regime_depth": OUT_DIR / "bottleneck_exposure_by_regime_depth.csv",
    "rank_promotion_by_reranker": OUT_DIR / "bottleneck_rank_promotion_by_reranker.csv",
    "rescue_harm_flags": OUT_DIR / "bottleneck_rescue_harm_flags.csv",
    "qchs_fallback_analysis": OUT_DIR / "bottleneck_qchs_fallback_analysis.csv",
    "candidate_overlap": OUT_DIR / "bottleneck_candidate_overlap.csv",
    "summary_by_reranker": OUT_DIR / "bottleneck_summary_by_reranker.csv",
    "query_level_parquet": OUT_DIR / "bottleneck_query_level_decomposition.parquet",
    "manifest": OUT_DIR / "run_manifest.json",
}


In [4]:
# ==== Helpers ====
def require_columns(df, required, label):
    missing = sorted(set(required).difference(df.columns))
    if missing:
        raise RuntimeError(f"{label} is missing required columns: {missing}")


def require_unique_columns(df, label):
    duplicated = df.columns[df.columns.duplicated()].astype(str).tolist()
    if duplicated:
        raise RuntimeError(f"{label} has duplicated columns: {duplicated}")


def boolean_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype("boolean")
    text = series.astype("string").str.strip().str.lower()
    numeric = pd.to_numeric(series, errors="coerce")
    mapped = text.map({
        "true": True, "false": False, "yes": True, "no": False,
        "1": True, "0": False,
    })
    return mapped.where(mapped.notna(), numeric.map({1.0: True, 0.0: False})).astype("boolean")


def metric_from_rank(rank_values, metric_name, cutoff):
    rank = pd.to_numeric(pd.Series(rank_values), errors="coerce").astype("float64")
    hit = rank.notna() & rank.le(int(cutoff))
    if metric_name == "NDCG":
        return pd.Series(
            np.where(hit, 1.0 / np.log2(rank.fillna(np.inf) + 1.0), 0.0),
            index=rank.index,
            dtype=float,
        )
    if metric_name == "MRR":
        return pd.Series(
            np.where(hit, 1.0 / rank.fillna(np.inf), 0.0),
            index=rank.index,
            dtype=float,
        )
    if metric_name == "HitRate":
        return hit.astype(float)
    raise ValueError(metric_name)


def build_rank_view(raw_df):
    key = [
        "case_id", "stage_condition", "method_family", "reranker_method",
        "candidate_pool_depth",
    ]
    invariant_columns = [
        "query_id", "user_id", "regime", "target_parent_asin",
        "target_exposed", "target_rank", "candidate_count", "candidate_source",
        "retrieval_method", "qchs_profile_available",
        "profile_fallback_flag", "profile_fallback_reason",
    ]
    variation = raw_df.groupby(key, dropna=False, observed=True)[
        invariant_columns
    ].nunique(dropna=False)
    bad = variation.gt(1).any(axis=1)
    if bad.any():
        raise RuntimeError(
            "Rank/provenance fields vary across metric rows: "
            + json.dumps(variation.loc[bad].head(10).reset_index().to_dict("records"))
        )
    view = (
        raw_df.sort_values(key, kind="mergesort")
        .drop_duplicates(key)[key + invariant_columns]
        .reset_index(drop=True)
    )
    view["target_exposed"] = boolean_series(view["target_exposed"]).fillna(False)
    view["qchs_profile_available"] = boolean_series(view["qchs_profile_available"])
    view["profile_fallback_flag"] = boolean_series(view["profile_fallback_flag"])
    return view


def exact_case_universe(left, right, label):
    left_cases = set(left["case_id"].astype(str))
    right_cases = set(right["case_id"].astype(str))
    if left_cases != right_cases:
        raise RuntimeError(
            f"{label} case-universe mismatch: "
            f"left_only={len(left_cases - right_cases)}, "
            f"right_only={len(right_cases - left_cases)}"
        )


def pair_rank_conditions(rank_df, family, left_condition, right_condition, contrast_name):
    left = rank_df.loc[
        rank_df["stage_condition"].eq(left_condition)
        & (
            rank_df["method_family"].eq(family)
            if left_condition not in {"P0", "P1-only"}
            else rank_df["method_family"].eq("shared_retrieval_baseline")
        )
    ].copy()
    right = rank_df.loc[
        rank_df["stage_condition"].eq(right_condition)
        & (
            rank_df["method_family"].eq(family)
            if right_condition not in {"P0", "P1-only"}
            else rank_df["method_family"].eq("shared_retrieval_baseline")
        )
    ].copy()
    common_depths = sorted(
        set(left["candidate_pool_depth"].astype(int))
        .intersection(right["candidate_pool_depth"].astype(int))
    )
    frames = []
    for depth in common_depths:
        left_depth = left.loc[left["candidate_pool_depth"].eq(depth)].copy()
        right_depth = right.loc[right["candidate_pool_depth"].eq(depth)].copy()
        exact_case_universe(left_depth, right_depth, f"{contrast_name}@{depth}")
        merged = left_depth.merge(
            right_depth,
            on=["case_id", "candidate_pool_depth"],
            how="inner",
            suffixes=("_left", "_right"),
            validate="one_to_one",
        )
        if not merged["regime_left"].astype(str).eq(merged["regime_right"].astype(str)).all():
            raise RuntimeError(f"Regime mismatch in {contrast_name}@{depth}.")
        merged["category_id"] = CATEGORY_ID
        merged["category_key"] = CATEGORY_KEY
        merged["category_label"] = CATEGORY_LABEL
        merged["reranker_family"] = family
        merged["contrast_name"] = contrast_name
        merged["left_condition"] = left_condition
        merged["right_condition"] = right_condition
        merged["regime"] = merged["regime_right"].astype(str)
        merged["left_target_exposed"] = boolean_series(
            merged["target_exposed_left"]
        ).fillna(False)
        merged["right_target_exposed"] = boolean_series(
            merged["target_exposed_right"]
        ).fillna(False)
        merged["both_exposed"] = (
            merged["left_target_exposed"] & merged["right_target_exposed"]
        )
        merged["exposure_rescue"] = (
            ~merged["left_target_exposed"] & merged["right_target_exposed"]
        )
        merged["exposure_loss"] = (
            merged["left_target_exposed"] & ~merged["right_target_exposed"]
        )
        left_rank = pd.to_numeric(merged["target_rank_left"], errors="coerce")
        right_rank = pd.to_numeric(merged["target_rank_right"], errors="coerce")
        merged["rank_improvement"] = np.where(
            merged["both_exposed"], left_rank - right_rank, np.nan
        )
        merged["ndcg_at_5_difference"] = (
            metric_from_rank(right_rank, "NDCG", PRIMARY_PROMOTION_CUTOFF).to_numpy()
            - metric_from_rank(left_rank, "NDCG", PRIMARY_PROMOTION_CUTOFF).to_numpy()
        )
        merged["mrr_at_5_difference"] = (
            metric_from_rank(right_rank, "MRR", PRIMARY_PROMOTION_CUTOFF).to_numpy()
            - metric_from_rank(left_rank, "MRR", PRIMARY_PROMOTION_CUTOFF).to_numpy()
        )
        merged["improved"] = merged["both_exposed"] & merged["rank_improvement"].gt(0)
        merged["unchanged"] = merged["both_exposed"] & merged["rank_improvement"].eq(0)
        merged["worsened"] = merged["both_exposed"] & merged["rank_improvement"].lt(0)
        frames.append(merged)
    if not frames:
        # pd.concat([]) raises an opaque "No objects to concatenate". Any pair that
        # touches a design-excluded cell -- ('shannon', 'b02') in the executed
        # Notebook 14 registry -- or that shares no candidate pool depth must fail
        # with an attributable message instead.
        raise RuntimeError(
            f"{contrast_name}: the pair is empty in the Notebook 14 canonical frame "
            f"(family={family!r}, left={left_condition!r}, right={right_condition!r}). "
            "Check the design-exclusion registry before pairing this cell."
        )
    return pd.concat(frames, ignore_index=True, sort=False)


def brand_total_variation(brands_left, brands_right):
    left = Counter(str(value) for value in brands_left)
    right = Counter(str(value) for value in brands_right)
    left_total = sum(left.values())
    right_total = sum(right.values())
    if left_total == 0 or right_total == 0:
        return np.nan
    keys = set(left).union(right)
    return float(
        0.5 * sum(
            abs(left.get(key, 0) / left_total - right.get(key, 0) / right_total)
            for key in keys
        )
    )


def json_records_sample(df, columns, n=10):
    if df.empty:
        return "[]"
    return json.dumps(
        df.loc[:, [column for column in columns if column in df.columns]]
        .head(n)
        .to_dict("records"),
        ensure_ascii=False,
    )


In [5]:
# ==== Load Final Notebook 14 Canonical Rank Contract ====
required_inputs = [PIPELINE_MANIFEST_PATH, CANONICAL_RAW_PATH, NB14_AVAILABILITY_PATH]
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise RuntimeError(f"Missing final Notebook 14 artifacts: {missing_inputs}")

pipeline_manifest = json.loads(PIPELINE_MANIFEST_PATH.read_text(encoding="utf-8"))
canonical_raw_df = pd.read_parquet(CANONICAL_RAW_PATH)
method_availability_df = pd.read_csv(NB14_AVAILABILITY_PATH)

raw_required = [
    "case_id", "query_id", "user_id", "regime", "stage_condition",
    "reranker_method", "method_family", "candidate_pool_depth",
    "metric_name", "metric_cutoff", "metric_value", "target_parent_asin",
    "target_exposed", "target_rank", "candidate_count", "candidate_source",
    "retrieval_method", "qchs_profile_available", "profile_fallback_flag",
    "profile_fallback_reason",
]
require_columns(canonical_raw_df, raw_required, "Notebook 14 canonical raw data")
require_unique_columns(canonical_raw_df, "Notebook 14 canonical raw data")

canonical_key = [
    "case_id", "stage_condition", "reranker_method",
    "candidate_pool_depth", "metric_name", "metric_cutoff",
]
if canonical_raw_df.duplicated(canonical_key).any():
    raise RuntimeError("Notebook 14 canonical raw data contain duplicated keys.")

rank_view_df = build_rank_view(canonical_raw_df)

# ---- design-exclusion registry, read from the producer, never re-derived ----
# The executed Notebook 14 publishes exactly one structurally undefined cell:
# ('shannon', 'b02'). Shannon is training-free, so a retrieval-personalised
# no-prior *re-fit* of it does not exist as an object of study. Completeness must
# therefore be asserted per observed design cell against this registry, and never
# as the orthogonal family x condition product.
def parse_design_excluded_pair(pair):
    if isinstance(pair, dict):
        family = pair.get("reranker_family", pair.get("method_family", pair.get("family")))
        condition = pair.get("stage_condition", pair.get("condition"))
    else:
        try:
            family, condition = pair[0], pair[1]
        except Exception as exc:
            raise RuntimeError(
                f"Unsupported design-exclusion manifest row: {pair!r}"
            ) from exc
    if family is None or condition is None:
        raise RuntimeError(f"Incomplete design-exclusion manifest row: {pair!r}")
    return str(family), str(condition)

_design_excluded_rows = (
    (pipeline_manifest.get("structural_unavailable_cells") or {})
    .get("design_excluded_method_condition_pairs", [])
)
DESIGN_EXCLUDED_METHOD_CONDITION_PAIRS = {
    parse_design_excluded_pair(pair) for pair in _design_excluded_rows
}
if not DESIGN_EXCLUDED_METHOD_CONDITION_PAIRS:
    raise RuntimeError(
        "Notebook 14 manifest publishes no design-exclusion registry. The "
        "(shannon, b02) hole is a design fact, not an accident, and its absence "
        "from the manifest means the upstream artefact is not the executed one."
    )
print(
    "Design-excluded (family, condition) pairs:",
    sorted(DESIGN_EXCLUDED_METHOD_CONDITION_PAIRS),
)
observed_families = set(
    rank_view_df.loc[
        rank_view_df["stage_condition"].isin(PROMOTION_LADDER_CONDITIONS),
        "method_family",
    ].astype(str)
)
if observed_families != set(RERANKER_FAMILIES):
    raise RuntimeError(f"Missing reranker families: {sorted(set(RERANKER_FAMILIES) - observed_families)}")
gam_depths = set(
    rank_view_df.loc[
        rank_view_df["method_family"].eq("gam")
        & rank_view_df["stage_condition"].isin(PROMOTION_LADDER_CONDITIONS),
        "candidate_pool_depth",
    ].astype(int)
)
# The executed Notebook 14 fits GAM once at depth 1000 and then EVALUATES it on
# deterministic fixed prefixes at every candidate pool depth
# (manifest gam_depth_policy.evaluation_policy =
#  "fit_once_at_1000_and_evaluate_deterministic_fixed_prefixes";
#  validation_results["gam_fixed_prefix_depths_complete"] is True).
# Asserting gam_depths == {1000} therefore contradicts the artefact of record.
# The contract is read from the manifest rather than hard-coded, so the gate can
# never drift from the producer again.
_gam_policy = (pipeline_manifest.get("gam_depth_policy") or {})
GAM_EXPECTED_DEPTHS = set(
    int(depth)
    for depth in (
        _gam_policy.get("available_candidate_pool_depths")
        or (pipeline_manifest.get("pool_depth_contract") or {}).get("candidate_pool_depths")
        or []
    )
)
if not GAM_EXPECTED_DEPTHS:
    raise RuntimeError(
        "Notebook 14 manifest publishes no GAM depth policy; the depth contract "
        "cannot be verified and must not be silently assumed."
    )
if gam_depths != GAM_EXPECTED_DEPTHS:
    raise RuntimeError(
        f"GAM depth contract mismatch: observed={sorted(gam_depths)}, "
        f"manifest={sorted(GAM_EXPECTED_DEPTHS)}, "
        f"fit_depth={_gam_policy.get('fit_depth')}, "
        f"policy={_gam_policy.get('evaluation_policy')!r}"
    )


Design-excluded (family, condition) pairs: [('shannon', 'b02')]


In [6]:
# ==== Shared Stage 1 Exposure Transition and Reranker Promotion ====
p0_rank_df = rank_view_df.loc[
    rank_view_df["stage_condition"].eq("P0")
    & rank_view_df["reranker_method"].eq("none")
].copy()
p1_rank_df = rank_view_df.loc[
    rank_view_df["stage_condition"].eq("P1-only")
    & rank_view_df["reranker_method"].eq("none")
].copy()

transition_frames = []
for depth in sorted(set(p0_rank_df["candidate_pool_depth"]).intersection(
    set(p1_rank_df["candidate_pool_depth"])
)):
    p0_depth = p0_rank_df.loc[p0_rank_df["candidate_pool_depth"].eq(depth)].copy()
    p1_depth = p1_rank_df.loc[p1_rank_df["candidate_pool_depth"].eq(depth)].copy()
    exact_case_universe(p0_depth, p1_depth, f"P0_vs_P1-only@{depth}")
    merged = p0_depth.merge(
        p1_depth,
        on=["case_id", "candidate_pool_depth"],
        how="inner",
        suffixes=("_p0", "_p1"),
        validate="one_to_one",
    )
    if not merged["regime_p0"].astype(str).eq(merged["regime_p1"].astype(str)).all():
        raise RuntimeError(f"Stage 1 regime mismatch at depth={depth}.")
    p0_exposed = boolean_series(merged["target_exposed_p0"]).fillna(False)
    p1_exposed = boolean_series(merged["target_exposed_p1"]).fillna(False)
    merged["category_id"] = CATEGORY_ID
    merged["category_key"] = CATEGORY_KEY
    merged["category_label"] = CATEGORY_LABEL
    merged["regime"] = merged["regime_p1"].astype(str)
    merged["p0_target_exposed"] = p0_exposed
    merged["p1_only_target_exposed"] = p1_exposed
    merged["stage1_exposure_rescue"] = ~p0_exposed & p1_exposed
    merged["stage1_exposure_loss"] = p0_exposed & ~p1_exposed
    merged["stage1_exposure_transition"] = np.select(
        [
            ~p0_exposed & ~p1_exposed,
            p0_exposed & ~p1_exposed,
            ~p0_exposed & p1_exposed,
            p0_exposed & p1_exposed,
        ],
        ["neither", "P0_only", "P1-only_only", "both"],
        default="invalid",
    )
    merged["interpretation_label"] = merged["stage1_exposure_transition"].map({
        "P1-only_only": "exposure_rescue",
        "P0_only": "exposure_loss",
        "both": "both_exposed_for_rank_comparison",
        "neither": "unresolved_stage1_exposure_failure",
    })
    merged["p0_target_rank"] = pd.to_numeric(merged["target_rank_p0"], errors="coerce")
    merged["p1_only_target_rank"] = pd.to_numeric(merged["target_rank_p1"], errors="coerce")
    merged["target_rank_shift_p1_minus_p0"] = (
        merged["p1_only_target_rank"] - merged["p0_target_rank"]
    )
    merged["qchs_profile_available"] = boolean_series(
        merged["qchs_profile_available_p1"]
    )
    merged["profile_fallback_flag"] = boolean_series(
        merged["profile_fallback_flag_p1"]
    )
    transition_frames.append(merged)

bottleneck_exposure_transition_df = pd.concat(
    transition_frames, ignore_index=True, sort=False
)
transition_group = [
    "category_id", "category_key", "category_label", "regime",
    "candidate_pool_depth", "stage1_exposure_transition", "interpretation_label",
]
transition_counts = (
    bottleneck_exposure_transition_df
    .groupby(transition_group, dropna=False, observed=True)
    .agg(count=("case_id", "nunique"))
    .reset_index()
)
transition_totals = (
    bottleneck_exposure_transition_df
    .groupby(
        ["category_id", "category_key", "category_label", "regime", "candidate_pool_depth"],
        dropna=False,
        observed=True,
    )
    .agg(total_cases=("case_id", "nunique"))
    .reset_index()
)
bottleneck_exposure_by_regime_depth_df = transition_counts.merge(
    transition_totals,
    on=["category_id", "category_key", "category_label", "regime", "candidate_pool_depth"],
    how="left",
    validate="many_to_one",
)
bottleneck_exposure_by_regime_depth_df["rate"] = (
    bottleneck_exposure_by_regime_depth_df["count"]
    / bottleneck_exposure_by_regime_depth_df["total_cases"]
)

promotion_frames = []
for family in RERANKER_FAMILIES:
    promotion_frames.extend([
        pair_rank_conditions(
            rank_view_df, family, "P2-Q", "P2-P", "P2-P_vs_P2-Q"
        ),
        pair_rank_conditions(
            rank_view_df, family, "P2-P", "Full", "Full_vs_P2-P"
        ),
    ])
bottleneck_rank_promotion_by_reranker_df = pd.concat(
    promotion_frames, ignore_index=True, sort=False
)


In [7]:
# ==== Notebook 09 Candidate Overlap and Fallback Identity ====
candidate_overlap_columns = [
    "category_id", "category_key", "category_label", "case_id", "regime",
    "candidate_pool_depth", "p0_candidate_count", "p1_candidate_count",
    "intersection_count", "union_count", "jaccard", "overlap_at_k_count",
    "overlap_at_k_rate", "candidates_added_by_personalization",
    "candidates_removed_by_personalization", "candidate_order_identical",
    "p0_target_rank", "p1_target_rank", "target_rank_shift_p1_minus_p0",
    "candidate_brand_total_variation", "profile_matched_candidate_share",
    "profile_match_contract_status", "qchs_profile_available",
    "profile_fallback_flag",
]
candidate_overlap_available = False
stage1_manifest = {}
p0_candidate_path = None
p1_candidate_path = None
profile_match_candidate_columns = [
    "profile_matched_candidate",
    "candidate_profile_match",
    "profile_match_flag",
    "functional_profile_match",
    "qchs_candidate_profile_match",
]

if STAGE1_MANIFEST_PATH.exists():
    stage1_manifest = json.loads(STAGE1_MANIFEST_PATH.read_text(encoding="utf-8"))
    output_paths = stage1_manifest.get("output_paths", {})
    if {
        "query_only_winner_long", "personalized_winner_long"
    }.issubset(output_paths):
        p0_candidate_path = Path(output_paths["query_only_winner_long"])
        p1_candidate_path = Path(output_paths["personalized_winner_long"])
        candidate_overlap_available = (
            p0_candidate_path.exists() and p1_candidate_path.exists()
        )

candidate_overlap_rows = []
if candidate_overlap_available:
    p0_candidates_df = pd.read_parquet(p0_candidate_path)
    p1_candidates_df = pd.read_parquet(p1_candidate_path)

    # Accept older Notebook 09 exports without changing upstream artifacts.
    candidate_column_aliases = {
        "candidate_brand_raw": "candidate_brand_facet_text",
        "qcha_profile_available": "qchs_profile_available",
        "qcha_fallback_flag": "profile_fallback_flag",
        "qchs_fallback_flag": "profile_fallback_flag",
    }
    for _candidate_frame in [p0_candidates_df, p1_candidates_df]:
        for old_name, canonical_name in candidate_column_aliases.items():
            if canonical_name not in _candidate_frame.columns and old_name in _candidate_frame.columns:
                _candidate_frame[canonical_name] = _candidate_frame[old_name]

    if "qchs_profile_available" not in p1_candidates_df.columns and "profile_fallback_flag" in p1_candidates_df.columns:
        p1_candidates_df["qchs_profile_available"] = ~boolean_series(
            p1_candidates_df["profile_fallback_flag"]
        ).fillna(False)
    if "profile_fallback_flag" not in p1_candidates_df.columns and "qchs_profile_available" in p1_candidates_df.columns:
        p1_candidates_df["profile_fallback_flag"] = ~boolean_series(
            p1_candidates_df["qchs_profile_available"]
        ).fillna(False)

    candidate_required = [
        "case_id", "regime", "candidate_parent_asin", "candidate_rank",
        "candidate_brand_facet_text", "is_gt",
    ]
    require_columns(p0_candidates_df, candidate_required, "Notebook 09 P0 candidates")
    require_columns(
        p1_candidates_df,
        [
            *candidate_required,
            "qchs_profile_available", "profile_fallback_flag",
        ],
        "Notebook 09 P1 candidates",
    )
    for frame in [p0_candidates_df, p1_candidates_df]:
        frame["case_id"] = frame["case_id"].astype(str)
        frame["candidate_parent_asin"] = frame["candidate_parent_asin"].astype(str)
        frame["candidate_rank"] = pd.to_numeric(
            frame["candidate_rank"], errors="raise"
        ).astype(int)
        frame["is_gt"] = pd.to_numeric(frame["is_gt"], errors="raise").astype(int)
    if p0_candidates_df.duplicated(["case_id", "candidate_parent_asin"]).any():
        raise RuntimeError("Notebook 09 P0 candidates contain duplicated case/candidate rows.")
    if p1_candidates_df.duplicated(["case_id", "candidate_parent_asin"]).any():
        raise RuntimeError("Notebook 09 P1 candidates contain duplicated case/candidate rows.")

    overlap_depths = pipeline_manifest["pool_depth_contract"]["candidate_pool_depths"]
    for depth in overlap_depths:
        p0_depth = p0_candidates_df.loc[
            p0_candidates_df["candidate_rank"].le(int(depth))
        ].copy()
        p1_depth = p1_candidates_df.loc[
            p1_candidates_df["candidate_rank"].le(int(depth))
        ].copy()
        p0_counts = p0_depth.groupby("case_id").size()
        p1_counts = p1_depth.groupby("case_id").size()
        if not p0_counts.eq(int(depth)).all() or not p1_counts.eq(int(depth)).all():
            raise RuntimeError(f"Notebook 09 candidate prefix is not exact-K at depth={depth}.")
        if set(p0_counts.index) != set(p1_counts.index):
            raise RuntimeError(f"Notebook 09 P0/P1 case universes differ at depth={depth}.")

        p0_groups = {
            case_id: group.sort_values("candidate_rank", kind="mergesort")
            for case_id, group in p0_depth.groupby("case_id", sort=False)
        }
        p1_groups = {
            case_id: group.sort_values("candidate_rank", kind="mergesort")
            for case_id, group in p1_depth.groupby("case_id", sort=False)
        }
        for case_id in sorted(p0_groups):
            p0_group = p0_groups[case_id]
            p1_group = p1_groups[case_id]
            p0_items = p0_group["candidate_parent_asin"].astype(str).tolist()
            p1_items = p1_group["candidate_parent_asin"].astype(str).tolist()
            p0_set = set(p0_items)
            p1_set = set(p1_items)
            intersection_count = len(p0_set.intersection(p1_set))
            union_count = len(p0_set.union(p1_set))
            p0_target = p0_group.loc[p0_group["is_gt"].eq(1), "candidate_rank"]
            p1_target = p1_group.loc[p1_group["is_gt"].eq(1), "candidate_rank"]
            p0_target_rank = int(p0_target.min()) if not p0_target.empty else np.nan
            p1_target_rank = int(p1_target.min()) if not p1_target.empty else np.nan
            candidate_overlap_rows.append({
                "category_id": CATEGORY_ID,
                "category_key": CATEGORY_KEY,
                "category_label": CATEGORY_LABEL,
                "case_id": str(case_id),
                "regime": str(p1_group["regime"].iloc[0]),
                "candidate_pool_depth": int(depth),
                "p0_candidate_count": len(p0_items),
                "p1_candidate_count": len(p1_items),
                "intersection_count": intersection_count,
                "union_count": union_count,
                "jaccard": float(intersection_count / union_count) if union_count else np.nan,
                "overlap_at_k_count": intersection_count,
                "overlap_at_k_rate": float(intersection_count / int(depth)),
                "candidates_added_by_personalization": len(p1_set - p0_set),
                "candidates_removed_by_personalization": len(p0_set - p1_set),
                "candidate_order_identical": p0_items == p1_items,
                "p0_target_rank": p0_target_rank,
                "p1_target_rank": p1_target_rank,
                "target_rank_shift_p1_minus_p0": p1_target_rank - p0_target_rank,
                "candidate_brand_total_variation": brand_total_variation(
                    p0_group["candidate_brand_facet_text"],
                    p1_group["candidate_brand_facet_text"],
                ),
                "profile_matched_candidate_share": (
                    float(boolean_series(p1_group[profile_match_column]).mean())
                    if (profile_match_column := next((column for column in profile_match_candidate_columns if column in p1_group.columns), None))
                    else np.nan
                ),
                "profile_match_contract_status": (
                    f"available:{profile_match_column}"
                    if profile_match_column
                    else "not_available_in_notebook09"
                ),
                "qchs_profile_available": bool(
                    boolean_series(p1_group["qchs_profile_available"]).iloc[0]
                ),
                "profile_fallback_flag": bool(
                    boolean_series(p1_group["profile_fallback_flag"]).iloc[0]
                ),
            })

bottleneck_candidate_overlap_df = pd.DataFrame(
    candidate_overlap_rows, columns=candidate_overlap_columns
)
if candidate_overlap_available:
    fallback_overlap = bottleneck_candidate_overlap_df.loc[
        bottleneck_candidate_overlap_df["profile_fallback_flag"].astype(bool)
    ]
    if not fallback_overlap["candidate_order_identical"].all():
        sample = fallback_overlap.loc[
            ~fallback_overlap["candidate_order_identical"]
        ].head(10)
        raise RuntimeError(
            "Notebook 09 fallback candidates do not preserve exact P0 order: "
            + json_records_sample(
                sample, ["case_id", "regime", "candidate_pool_depth"]
            )
        )


In [8]:
# ==== Rescue/harm flags, Fallback diagnostics, and Summaries ====
required_upstream_frames = [
    "bottleneck_exposure_transition_df",
    "bottleneck_rank_promotion_by_reranker_df",
]
missing_upstream_frames = [
    name for name in required_upstream_frames if name not in globals()
]
if missing_upstream_frames:
    raise RuntimeError(
        "Run the previous cell '# Shared Stage 1 exposure transition and reranker promotion' "
        "before this cell. "
        f"Missing: {missing_upstream_frames}"
    )

flag_frames = []
transition_flag_columns = [
    "case_id", "regime", "candidate_pool_depth",
    "stage1_exposure_rescue", "stage1_exposure_loss",
    "qchs_profile_available", "profile_fallback_flag",
]
for family in RERANKER_FAMILIES:
    base = bottleneck_exposure_transition_df[transition_flag_columns].copy()
    p2_pair = bottleneck_rank_promotion_by_reranker_df.loc[
        bottleneck_rank_promotion_by_reranker_df["reranker_family"].eq(family)
        & bottleneck_rank_promotion_by_reranker_df["contrast_name"].eq(
            "P2-P_vs_P2-Q"
        )
    ][
        [
            "case_id", "candidate_pool_depth", "improved", "worsened",
            "unchanged", "rank_improvement",
        ]
    ].rename(columns={
        "improved": "stage2_prior_promotion_gain",
        "worsened": "stage2_prior_promotion_harm",
        "unchanged": "stage2_prior_promotion_unchanged",
        "rank_improvement": "stage2_prior_rank_improvement",
    })
    full_pair = bottleneck_rank_promotion_by_reranker_df.loc[
        bottleneck_rank_promotion_by_reranker_df["reranker_family"].eq(family)
        & bottleneck_rank_promotion_by_reranker_df["contrast_name"].eq(
            "Full_vs_P2-P"
        )
    ][
        [
            "case_id", "candidate_pool_depth", "exposure_rescue", "exposure_loss",
            "improved", "worsened", "unchanged", "rank_improvement",
        ]
    ].rename(columns={
        "exposure_rescue": "full_exposure_rescue",
        "exposure_loss": "full_exposure_loss",
        "improved": "full_rank_promotion_gain",
        "worsened": "full_rank_promotion_harm",
        "unchanged": "full_rank_promotion_unchanged",
        "rank_improvement": "full_rank_improvement",
    })
    flags = base.merge(
        p2_pair,
        on=["case_id", "candidate_pool_depth"],
        how="left",
        validate="one_to_one",
    ).merge(
        full_pair,
        on=["case_id", "candidate_pool_depth"],
        how="left",
        validate="one_to_one",
    )
    boolean_flag_columns = [
        "stage1_exposure_rescue", "stage1_exposure_loss",
        "stage2_prior_promotion_gain", "stage2_prior_promotion_harm",
        "full_exposure_rescue", "full_exposure_loss",
        "full_rank_promotion_gain", "full_rank_promotion_harm",
    ]
    for column in boolean_flag_columns:
        flags[column] = flags[column].fillna(False).astype(bool)
    flags["no_measurable_change"] = ~flags[boolean_flag_columns].any(axis=1)
    flags["category_id"] = CATEGORY_ID
    flags["category_key"] = CATEGORY_KEY
    flags["category_label"] = CATEGORY_LABEL
    flags["reranker_family"] = family
    flags["shared_stage1_flag_repeated_for_display"] = True
    flag_frames.append(flags)
bottleneck_rescue_harm_flags_df = pd.concat(
    flag_frames, ignore_index=True, sort=False
)

source_inventory_df = pd.DataFrame(pipeline_manifest.get("source_inventory", []))
model_contract_required_families = [
    family for family in RERANKER_FAMILIES if family != "shannon"
]
required_model_contract_pairs = {
    (family, condition)
    for family in model_contract_required_families
    for condition in ["P2-P", "Full"]
}
observed_model_contract_pairs = set()
if not source_inventory_df.empty and {
    "method_family", "condition", "model_contract_sha256_json"
}.issubset(source_inventory_df.columns):
    contract_inventory = source_inventory_df.copy()
    contract_inventory["model_contract_sha256_json"] = (
        contract_inventory["model_contract_sha256_json"].fillna("").astype(str)
    )
    contract_inventory = contract_inventory.loc[
        contract_inventory["method_family"].astype(str).isin(model_contract_required_families)
        & contract_inventory["condition"].astype(str).isin(["P2-P", "Full"])
        & ~contract_inventory["model_contract_sha256_json"].isin(["", "[]", "null"])
    ].copy()
    observed_model_contract_pairs = set(
        zip(
            contract_inventory["method_family"].astype(str),
            contract_inventory["condition"].astype(str),
        )
    )
stage2_model_contract_verified = required_model_contract_pairs.issubset(
    observed_model_contract_pairs
)
stage2_model_contract_status = (
    "self_pool_model_contract_hashes_recorded_in_notebook14"
    if stage2_model_contract_verified
    else "missing_notebook14_model_contract_hashes"
)

fallback_case_ids = set(
    bottleneck_exposure_transition_df.loc[
        bottleneck_exposure_transition_df["profile_fallback_flag"].fillna(False),
        "case_id",
    ].astype(str)
)
fallback_rows = []
for family in RERANKER_FAMILIES:
    left = canonical_raw_df.loc[
        canonical_raw_df["stage_condition"].eq("P2-P")
        & canonical_raw_df["method_family"].eq(family)
    ].copy()
    right = canonical_raw_df.loc[
        canonical_raw_df["stage_condition"].eq("Full")
        & canonical_raw_df["method_family"].eq(family)
    ].copy()
    keys = ["case_id", "candidate_pool_depth", "metric_name", "metric_cutoff"]
    common = left.merge(
        right,
        on=keys,
        how="inner",
        suffixes=("_s2p", "_full"),
        validate="one_to_one",
    )
    common = common.loc[common["case_id"].astype(str).isin(fallback_case_ids)].copy()
    if common.empty:
        continue
    common["metric_difference_full_minus_s2p"] = (
        pd.to_numeric(common["metric_value_full"], errors="raise")
        - pd.to_numeric(common["metric_value_s2p"], errors="raise")
    )
    if candidate_overlap_available:
        common = common.merge(
            bottleneck_candidate_overlap_df[
                ["case_id", "candidate_pool_depth", "candidate_order_identical"]
            ],
            on=["case_id", "candidate_pool_depth"],
            how="left",
            validate="many_to_one",
        )
    else:
        common["candidate_order_identical"] = pd.NA
    common["regime"] = common["regime_full"].astype(str)

    scopes = []
    combination_columns = [
        "candidate_pool_depth", "metric_name", "metric_cutoff"
    ]
    for (depth, metric_name, metric_cutoff), combination in common.groupby(
        combination_columns, dropna=False, observed=True
    ):
        scopes.append((
            int(depth), str(metric_name), int(metric_cutoff),
            "overall", "overall", combination,
        ))
        scopes.extend(
            (
                int(depth), str(metric_name), int(metric_cutoff),
                "regime", str(regime), sub,
            )
            for regime, sub in combination.groupby(
                "regime", dropna=False, observed=True
            )
        )
    for depth, metric_name, metric_cutoff, user_scope, regime, sub in scopes:
        nonzero = sub.loc[
            ~np.isclose(
                sub["metric_difference_full_minus_s2p"].to_numpy(dtype=float),
                0.0,
                rtol=0.0,
                atol=0.0,
            )
        ]
        fallback_rows.append({
            "category_id": CATEGORY_ID,
            "category_key": CATEGORY_KEY,
            "category_label": CATEGORY_LABEL,
            "reranker_family": family,
            "candidate_pool_depth": depth,
            "metric_name": metric_name,
            "metric_cutoff": metric_cutoff,
            "user_scope": user_scope,
            "regime": regime,
            "fallback_case_count": int(sub["case_id"].nunique()),
            "fallback_rate_within_scope": float(
                sub["case_id"].nunique()
                / max(
                    bottleneck_exposure_transition_df.loc[
                        bottleneck_exposure_transition_df[
                            "candidate_pool_depth"
                        ].eq(depth)
                        & (
                            bottleneck_exposure_transition_df["regime"].astype(str).eq(regime)
                            if user_scope == "regime"
                            else True
                        ),
                        "case_id",
                    ].nunique(),
                    1,
                )
            ),
            "fallback_target_exposure_rate_full": float(
                boolean_series(sub["target_exposed_full"]).fillna(False).mean()
            ),
            "mean_metric_difference_full_minus_s2p": float(
                sub["metric_difference_full_minus_s2p"].mean()
            ),
            "max_abs_metric_difference": float(
                sub["metric_difference_full_minus_s2p"].abs().max()
            ),
            "nonzero_difference_count": int(nonzero["case_id"].nunique()),
            "nonzero_difference_sample": json_records_sample(
                nonzero,
                [
                    "case_id", "regime", "candidate_pool_depth",
                    "metric_name", "metric_cutoff",
                    "metric_value_s2p", "metric_value_full",
                    "metric_difference_full_minus_s2p",
                ],
            ),
            "p0_p1_candidate_order_identical": (
                bool(boolean_series(sub["candidate_order_identical"]).fillna(False).all())
                if candidate_overlap_available else pd.NA
            ),
            "full_s2p_prererank_candidate_identity_verified": (
                bool(boolean_series(sub["candidate_order_identical"]).fillna(False).all())
                if candidate_overlap_available else False
            ),
            "stage2_model_contract_verified": stage2_model_contract_verified,
            "stage2_model_contract_status": stage2_model_contract_status,
            # Full and RankP are separate fitted policies on different candidate interfaces.
        # Identical Stage-2 ranks are therefore not required on non-cold QCHS-fallback
        # cases, even though the Stage-1 candidate order is identical. This comparison
        # remains a non-blocking diagnostic.
        "expected_zero_if_candidate_and_model_contracts_identical": False,
            "contract_mismatch_detected": int(nonzero["case_id"].nunique()) > 0,
        })

fallback_output_columns = [
    "category_id", "category_key", "category_label", "reranker_family",
    "candidate_pool_depth", "metric_name", "metric_cutoff", "user_scope",
    "regime", "fallback_case_count", "fallback_rate_within_scope",
    "fallback_target_exposure_rate_full",
    "mean_metric_difference_full_minus_s2p", "max_abs_metric_difference",
    "nonzero_difference_count", "nonzero_difference_sample",
    "p0_p1_candidate_order_identical",
    "full_s2p_prererank_candidate_identity_verified",
    "stage2_model_contract_verified", "stage2_model_contract_status",
    "expected_zero_if_candidate_and_model_contracts_identical",
    "contract_mismatch_detected",
]
bottleneck_qchs_fallback_analysis_df = pd.DataFrame(
    fallback_rows, columns=fallback_output_columns
)

summary_rows = []
for (family, contrast_name, depth), group in bottleneck_rank_promotion_by_reranker_df.groupby(
    ["reranker_family", "contrast_name", "candidate_pool_depth"],
    dropna=False,
    observed=True,
):
    scopes = [("overall", "overall", group)]
    scopes.extend(
        ("regime", str(regime), sub)
        for regime, sub in group.groupby("regime", dropna=False, observed=True)
    )
    for user_scope, regime, sub in scopes:
        summary_rows.append({
            "category_id": CATEGORY_ID,
            "category_key": CATEGORY_KEY,
            "category_label": CATEGORY_LABEL,
            "reranker_family": family,
            "contrast_name": contrast_name,
            "candidate_pool_depth": int(depth),
            "user_scope": user_scope,
            "regime": regime,
            "case_count": int(sub["case_id"].nunique()),
            "both_exposed_count": int(sub["both_exposed"].sum()),
            "exposure_rescue_count": int(sub["exposure_rescue"].sum()),
            "exposure_loss_count": int(sub["exposure_loss"].sum()),
            "rank_improved_count": int(sub["improved"].sum()),
            "rank_unchanged_count": int(sub["unchanged"].sum()),
            "rank_worsened_count": int(sub["worsened"].sum()),
            "mean_rank_improvement_both_exposed": float(
                sub.loc[sub["both_exposed"], "rank_improvement"].mean()
            ),
            "mean_ndcg_at_5_difference": float(sub["ndcg_at_5_difference"].mean()),
            "mean_mrr_at_5_difference": float(sub["mrr_at_5_difference"].mean()),
        })
bottleneck_summary_by_reranker_df = pd.DataFrame(summary_rows)

bottleneck_query_level_decomposition_df = bottleneck_rescue_harm_flags_df.copy()
if candidate_overlap_available:
    bottleneck_query_level_decomposition_df = (
        bottleneck_query_level_decomposition_df.merge(
            bottleneck_candidate_overlap_df,
            on=[
                "category_id", "category_key", "category_label",
                "case_id", "regime", "candidate_pool_depth",
            ],
            how="left",
            validate="many_to_one",
            suffixes=("", "_candidate"),
        )
    )


In [9]:
# ==== Final Validation and Exports ====
required_upstream_objects = [
    'bottleneck_exposure_transition_df',
    'bottleneck_exposure_by_regime_depth_df',
    'bottleneck_rank_promotion_by_reranker_df',
    'bottleneck_rescue_harm_flags_df',
    'bottleneck_qchs_fallback_analysis_df',
    'bottleneck_candidate_overlap_df',
    'bottleneck_summary_by_reranker_df',
    'bottleneck_query_level_decomposition_df'
]
missing_upstream_objects = [
    name for name in required_upstream_objects if name not in globals()
]
if missing_upstream_objects:
    raise RuntimeError(
        'Run the upstream Notebook 17 decomposition cells before final validation/export.'
        f" Missing: {missing_upstream_objects}"
    )

output_frames = {
    "exposure_transition": bottleneck_exposure_transition_df,
    "exposure_by_regime_depth": bottleneck_exposure_by_regime_depth_df,
    "rank_promotion_by_reranker": bottleneck_rank_promotion_by_reranker_df,
    "rescue_harm_flags": bottleneck_rescue_harm_flags_df,
    "qchs_fallback_analysis": bottleneck_qchs_fallback_analysis_df,
    "candidate_overlap": bottleneck_candidate_overlap_df,
    "summary_by_reranker": bottleneck_summary_by_reranker_df,
    "query_level_parquet": bottleneck_query_level_decomposition_df,
}
for name, frame in output_frames.items():
    require_unique_columns(frame, name)

if set(bottleneck_rank_promotion_by_reranker_df["reranker_family"]) != set(
    RERANKER_FAMILIES
):
    raise RuntimeError("Promotion decomposition dropped a reranker family.")
gam_promotion_depths = set(
    bottleneck_rank_promotion_by_reranker_df.loc[
        bottleneck_rank_promotion_by_reranker_df["reranker_family"].eq("gam"),
        "candidate_pool_depth",
    ].astype(int)
)
if gam_promotion_depths != GAM_EXPECTED_DEPTHS:
    raise RuntimeError(
        f"GAM promotion depth mismatch: observed={sorted(gam_promotion_depths)}, "
        f"manifest={sorted(GAM_EXPECTED_DEPTHS)}"
    )

if candidate_overlap_available:
    fallback_overlap = bottleneck_candidate_overlap_df.loc[
        bottleneck_candidate_overlap_df["profile_fallback_flag"].astype(bool)
    ]
    if len(fallback_overlap) != (
        len(fallback_case_ids)
        * len(pipeline_manifest["pool_depth_contract"]["candidate_pool_depths"])
    ):
        raise RuntimeError("QCHS fallback cases were lost from candidate-overlap analysis.")

bottleneck_exposure_transition_df.to_csv(
    OUTPUT_FILES["exposure_transition"], index=False, encoding="utf-8-sig"
)
bottleneck_exposure_by_regime_depth_df.to_csv(
    OUTPUT_FILES["exposure_by_regime_depth"], index=False, encoding="utf-8-sig"
)
bottleneck_rank_promotion_by_reranker_df.to_csv(
    OUTPUT_FILES["rank_promotion_by_reranker"], index=False, encoding="utf-8-sig"
)
bottleneck_rescue_harm_flags_df.to_csv(
    OUTPUT_FILES["rescue_harm_flags"], index=False, encoding="utf-8-sig"
)
bottleneck_qchs_fallback_analysis_df.to_csv(
    OUTPUT_FILES["qchs_fallback_analysis"], index=False, encoding="utf-8-sig"
)
bottleneck_candidate_overlap_df.to_csv(
    OUTPUT_FILES["candidate_overlap"], index=False, encoding="utf-8-sig"
)
bottleneck_summary_by_reranker_df.to_csv(
    OUTPUT_FILES["summary_by_reranker"], index=False, encoding="utf-8-sig"
)
bottleneck_query_level_decomposition_df.to_parquet(
    OUTPUT_FILES["query_level_parquet"], index=False
)

manifest = {
    "notebook_name": NOTEBOOK_NAME,
    "category_id": CATEGORY_ID,
    "category_key": CATEGORY_KEY,
    "category_label": CATEGORY_LABEL,
    "analysis_output_dir": str(OUT_DIR),
    "notebook14_manifest_path": str(PIPELINE_MANIFEST_PATH),
    "notebook14_canonical_raw_path": str(CANONICAL_RAW_PATH),
    "notebook09_manifest_path": str(STAGE1_MANIFEST_PATH),
    "notebook09_query_only_candidate_path": (
        str(p0_candidate_path) if p0_candidate_path is not None else None
    ),
    "notebook09_personalized_candidate_path": (
        str(p1_candidate_path) if p1_candidate_path is not None else None
    ),
    "candidate_overlap_available": bool(candidate_overlap_available),
    "candidate_pool_depths": pipeline_manifest["pool_depth_contract"][
        "candidate_pool_depths"
    ],
    "primary_promotion_cutoff": PRIMARY_PROMOTION_CUTOFF,
    "reranker_families": RERANKER_FAMILIES,
    "promotion_contrasts": ["P2-P_vs_P2-Q", "Full_vs_P2-P"],
    "rescue_harm_flags_are_nonexclusive": True,
    "shared_stage1_transition_computed_once": True,
    "qchs_fallback_case_count": int(len(fallback_case_ids)),
    "qchs_fallback_cases_retained": True,
    "stage2_model_contract_verified": stage2_model_contract_verified,
    "stage2_model_contract_status": stage2_model_contract_status,
    "profile_match_contract_status": (
        "not_available_in_notebook09"
        if candidate_overlap_available else "candidate_files_unavailable"
    ),
    "output_files": {name: str(path) for name, path in OUTPUT_FILES.items()},
    "validation_results": {
        "canonical_key_unique": True,
        "all_reranker_families_retained": True,
        "gam_depth_1000_only": True,
        "shared_stage1_exposure_computed_once": True,
        "paired_case_universes_equal": True,
        "fallback_cases_preserved": True,
        "candidate_overlap_computed_from_candidate_ids": bool(
            candidate_overlap_available
        ),
        "output_columns_unique": True,
    },
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
OUTPUT_FILES["manifest"].write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Stage bottleneck outputs written to:", OUT_DIR)
print("Candidate overlap available:", candidate_overlap_available)
print("Stage 2 model-contract status:", stage2_model_contract_status)


Stage bottleneck outputs written to: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/stage_bottleneck_decomposition
Candidate overlap available: True
Stage 2 model-contract status: missing_notebook14_model_contract_hashes


In [10]:
# ==== Post-hoc Opportunity Decomposition at the Top-Five Boundary ====
required_upstream_objects = [
    'rank_view_df',
    'bottleneck_rank_promotion_by_reranker_df'
]
missing_upstream_objects = [
    name for name in required_upstream_objects if name not in globals()
]
if missing_upstream_objects:
    raise RuntimeError(
        "Run '# Shared Stage 1 exposure transition and reranker promotion' before A1 opportunity decomposition."
        f" Missing: {missing_upstream_objects}"
    )

OPPORTUNITY_CUTOFF = 5
OPPORTUNITY_CONTRASTS = ["P2-P_vs_P2-Q", "Full_vs_P2-P", "Full_vs_P2-Q"]
POST_HOC_LABEL = "post_hoc_descriptive_no_outcome_based_search"

_opp_frames = [bottleneck_rank_promotion_by_reranker_df]
_fullq = pd.concat(
    [pair_rank_conditions(rank_view_df, fam, "P2-Q", "Full", "Full_vs_P2-Q")
     for fam in RERANKER_FAMILIES],
    ignore_index=True, sort=False,
)
_opp_frames.append(_fullq)
opp_source = pd.concat(_opp_frames, ignore_index=True, sort=False)
opp_source = opp_source.loc[opp_source["contrast_name"].isin(OPPORTUNITY_CONTRASTS)].copy()

_lr = pd.to_numeric(opp_source["target_rank_left"], errors="coerce")
_rr = pd.to_numeric(opp_source["target_rank_right"], errors="coerce")
_exp_l = opp_source["left_target_exposed"].fillna(False).astype(bool)
_exp_r = opp_source["right_target_exposed"].fillna(False).astype(bool)
_in5_l = _exp_l & _lr.le(OPPORTUNITY_CUTOFF)
_in5_r = _exp_r & _rr.le(OPPORTUNITY_CUTOFF)
_exp_any = (_exp_l | _exp_r)

opp_source["opportunity_bucket"] = np.select(
    [
        (~_exp_l) & (~_exp_r),
        (~_in5_l) & _in5_r,
        _in5_l & (~_in5_r),
        _in5_l & _in5_r & _rr.lt(_lr),
        _in5_l & _in5_r & _rr.gt(_lr),
    ],
    ["absent", "entered_top5", "exited_top5", "improved_within_top5", "worsened_within_top5"],
    default="present_rank_unchanged",
)
opp_source["is_informative"] = _exp_any.to_numpy()

BUCKET_ORDER = ["absent", "present_rank_unchanged", "entered_top5", "exited_top5",
                "improved_within_top5", "worsened_within_top5"]

def _opp_scopes(frame):
    yield ("overall", "overall", frame)
    for reg, sub in frame.groupby("regime", dropna=False, observed=True):
        yield ("regime", str(reg), sub)

opp_rows, net_rows = [], []
for (contrast, family, depth), grp in opp_source.groupby(
    ["contrast_name", "reranker_family", "candidate_pool_depth"], dropna=False, observed=True):
    for user_scope, regime, sub in _opp_scopes(grp):
        n_scope = int(sub["case_id"].nunique())
        n_inf = int(sub.loc[sub["is_informative"], "case_id"].nunique())
        counts = sub.groupby("opportunity_bucket")["case_id"].nunique()
        n_entered = int(counts.get("entered_top5", 0))
        n_exited = int(counts.get("exited_top5", 0))
        net_rows.append({
            "category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family,
            "candidate_pool_depth": int(depth), "user_scope": user_scope, "regime": regime,
            "n_cases": n_scope, "effective_informative_n": n_inf,
            "net_top5_entries_per_1000": (float((n_entered - n_exited) / n_scope * 1000.0) if n_scope else float("nan")),
            "post_hoc_label": POST_HOC_LABEL,
        })
        for bucket in BUCKET_ORDER:
            b = sub.loc[sub["opportunity_bucket"].eq(bucket)]
            opp_rows.append({
                "category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family,
                "candidate_pool_depth": int(depth), "user_scope": user_scope, "regime": regime,
                "opportunity_bucket": bucket,
                "n_cases": int(b["case_id"].nunique()),
                "pct_of_scope": (float(b["case_id"].nunique() / n_scope) if n_scope else float("nan")),
                "conditional_mean_ndcg_at_5_difference": (float(b["ndcg_at_5_difference"].mean()) if len(b) else float("nan")),
                "post_hoc_label": POST_HOC_LABEL,
            })

opportunity_decomposition_df = pd.DataFrame(opp_rows)
opportunity_net_entries_df = pd.DataFrame(net_rows)

# QC: buckets are mutually exclusive & exhaustive (bucket sum == scope N per group)
_chk = (opportunity_decomposition_df
        .groupby(["contrast_name","reranker_family","candidate_pool_depth","user_scope","regime"])["n_cases"].sum()
        .reset_index()
        .merge(opportunity_net_entries_df[["contrast_name","reranker_family","candidate_pool_depth","user_scope","regime","n_cases"]],
               on=["contrast_name","reranker_family","candidate_pool_depth","user_scope","regime"], suffixes=("_buckets","_scope")))
if not (_chk["n_cases_buckets"] == _chk["n_cases_scope"]).all():
    raise RuntimeError("A1 opportunity buckets are not exhaustive/exclusive.")

opportunity_decomposition_df.to_csv(OUT_DIR / f"opportunity_decomposition_{CATEGORY_ID}.csv", index=False, encoding="utf-8-sig")
opportunity_net_entries_df.to_csv(OUT_DIR / f"opportunity_net_entries_{CATEGORY_ID}.csv", index=False, encoding="utf-8-sig")
print("A1 opportunity decomposition written to:", OUT_DIR)
print("  headline depth=1000 bucket rows:", int(opportunity_decomposition_df["candidate_pool_depth"].eq(1000).sum()))


A1 opportunity decomposition written to: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/stage_bottleneck_decomposition
  headline depth=1000 bucket rows: 288


In [11]:
# ==== Post-hoc Effective Sample Size and Approximate MDE ====
required_upstream_objects = [
    'opp_source'
]
missing_upstream_objects = [
    name for name in required_upstream_objects if name not in globals()
]
if missing_upstream_objects:
    raise RuntimeError(
        'Run the A1 opportunity decomposition cell before A2 effective-n/MDE.'
        f" Missing: {missing_upstream_objects}"
    )

A2_Z_SUM_80 = 1.959963985 + 0.841621234  # z_{0.975} + z_{0.80}
A2_POST_HOC = "post_hoc_descriptive_no_outcome_based_search"

def _a2_row(frame, contrast, family, depth, scope, regime):
    d = pd.to_numeric(frame["ndcg_at_5_difference"], errors="coerce").dropna().to_numpy(dtype=float)
    n = int(len(d))
    n_inf = int(frame["is_informative"].fillna(False).astype(bool).sum())
    sd = float(np.std(d, ddof=1)) if n > 1 else float("nan")
    mde = float(A2_Z_SUM_80 * sd / np.sqrt(n)) if (n > 1 and np.isfinite(sd)) else float("nan")
    return {
        "category_id": CATEGORY_ID, "contrast_name": contrast, "reranker_family": family,
        "candidate_pool_depth": int(depth), "user_scope": scope, "regime": regime,
        "n_pairs": n, "effective_informative_n": n_inf,
        "mean_delta_ndcg_at_5": (float(np.mean(d)) if n else float("nan")),
        "sd_delta_ndcg_at_5": sd, "mde_80pct_power": mde,
        "z_sum_alpha05_power80": A2_Z_SUM_80, "post_hoc_label": A2_POST_HOC,
    }

_a2_rows = []
for (contrast, family, depth), grp in opp_source.groupby(
    ["contrast_name", "reranker_family", "candidate_pool_depth"], dropna=False, observed=True):
    _a2_rows.append(_a2_row(grp, contrast, family, depth, "overall", "overall"))
    for reg, sub in grp.groupby("regime", dropna=False, observed=True):
        _a2_rows.append(_a2_row(sub, contrast, family, depth, "regime", str(reg)))
effective_n_mde_df = pd.DataFrame(_a2_rows)
effective_n_mde_df.to_csv(OUT_DIR / f"effective_n_mde_{CATEGORY_ID}.csv", index=False, encoding="utf-8-sig")
print("A2 effective-n/MDE written to:", OUT_DIR, "| headline depth=1000 rows:",
      int(effective_n_mde_df["candidate_pool_depth"].eq(1000).sum()))


A2 effective-n/MDE written to: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/analysis/stage_bottleneck_decomposition | headline depth=1000 rows: 48


## Interaction I: Cross-Stage Departure from Additivity

Interaction I is:

\[
(\mathrm{Full-RankP})-(\mathrm{S1-P-S1-Q}).
\]

It compares the candidate-source increment under a prior-aware reranker with the change in raw Stage 1 ordering. Results are reported overall, by history regime, and for the QCHS-active subset across the five fixed candidate depths.

This is a post-hoc descriptive comparison. Its 2,000-replicate case/user bootstrap interval quantifies uncertainty, but no randomization test or multiplicity-adjusted decision is attached. An interval containing zero means that departure from additivity was not detected.


In [12]:
# ==== Post-hoc Interaction I: (Full - RankP) - (P1-only - P0) ====
required_upstream_objects = [
    'rank_view_df',
    'bottleneck_exposure_transition_df',
    'bottleneck_rank_promotion_by_reranker_df'
]
missing_upstream_objects = [
    name for name in required_upstream_objects if name not in globals()
]
if missing_upstream_objects:
    raise RuntimeError(
        "Run '# Shared Stage 1 exposure transition and reranker promotion' before NEW-I interaction."
        f" Missing: {missing_upstream_objects}"
    )

INTERACTION_POST_HOC = "exploratory_secondary_descriptive_no_outcome_based_search"
INTERACTION_BOOT_REPS = 2000
INTERACTION_SEED = 42

# (P1-only - P0) per case x depth : shared retrieval baseline, family-independent
_st1 = bottleneck_exposure_transition_df.copy()
_st1["ndcg5_p0"] = metric_from_rank(_st1["p0_target_rank"], "NDCG", PRIMARY_PROMOTION_CUTOFF).to_numpy()
_st1["ndcg5_p1only"] = metric_from_rank(_st1["p1_only_target_rank"], "NDCG", PRIMARY_PROMOTION_CUTOFF).to_numpy()
_st1["delta_stage1"] = _st1["ndcg5_p1only"] - _st1["ndcg5_p0"]
_st1_keyed = _st1[["case_id", "candidate_pool_depth", "delta_stage1", "qchs_profile_available"]].copy()
_st1_keyed["case_id"] = _st1_keyed["case_id"].astype(str)

# (Full - RankP) per case x depth x family
_st2 = bottleneck_rank_promotion_by_reranker_df.loc[
    bottleneck_rank_promotion_by_reranker_df["contrast_name"].eq("Full_vs_P2-P"),
    ["case_id", "candidate_pool_depth", "regime", "reranker_family", "ndcg_at_5_difference"],
].rename(columns={"ndcg_at_5_difference": "delta_full_minus_s2p"}).copy()
_st2["case_id"] = _st2["case_id"].astype(str)

# case -> user map (from P0 shared rows)
_umap = (rank_view_df.loc[rank_view_df["stage_condition"].eq("P0")]
         .drop_duplicates("case_id").set_index("case_id")["user_id"])

interaction_did = _st2.merge(
    _st1_keyed, on=["case_id", "candidate_pool_depth"], how="inner", validate="many_to_one")
interaction_did["did_interaction"] = (
    interaction_did["delta_full_minus_s2p"] - interaction_did["delta_stage1"])
interaction_did["user_id"] = interaction_did["case_id"].map(_umap)
interaction_did["qchs_active"] = boolean_series(interaction_did["qchs_profile_available"]).fillna(False).astype(bool)

_boot_rng = np.random.default_rng(INTERACTION_SEED)
def _cluster_boot_ci(frame, value_col):
    v = frame[[value_col, "user_id"]].dropna()
    if len(v) < 2:
        return (float("nan"), float("nan"))
    g = v.groupby("user_id")[value_col].agg(["sum", "count"])
    usum = g["sum"].to_numpy(dtype=float); ucnt = g["count"].to_numpy(dtype=float); G = len(usum)
    idx = _boot_rng.integers(0, G, size=(INTERACTION_BOOT_REPS, G))
    boot = usum[idx].sum(axis=1) / ucnt[idx].sum(axis=1)
    lo, hi = np.quantile(boot, [0.025, 0.975])
    return float(lo), float(hi)

def _verdict(mean_did, lo, hi):
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return "insufficient_n"
    if lo > 0:
        return "synergy_superadditive"
    if hi < 0:
        return "partial_substitution_subadditive"
    return "additive_complementarity_no_significant_synergy"

_rows = []
_subsets = [("full", interaction_did), ("qchs_active", interaction_did.loc[interaction_did["qchs_active"]])]
for _subset_name, _dsub in _subsets:
    for (family, depth) in sorted(
        {(f, int(d)) for f, d in _dsub[["reranker_family", "candidate_pool_depth"]].itertuples(index=False)}):
        _grp = _dsub.loc[_dsub["reranker_family"].eq(family) & _dsub["candidate_pool_depth"].eq(depth)]
        _scopes = [("overall", "overall", _grp)] + [
            ("regime", str(r), g) for r, g in sorted(_grp.groupby("regime", observed=True), key=lambda x: str(x[0]))]
        for _scope, _regime, _sub in _scopes:
            d = _sub["did_interaction"].to_numpy(dtype=float)
            n = int(len(d)); m = float(np.mean(d)) if n else float("nan")
            lo, hi = _cluster_boot_ci(_sub, "did_interaction")
            _rows.append({
                "category_id": CATEGORY_ID, "interaction": "(I) rerank x pool-personalization",
                "did_formula": "(Full-S2P)-(P1only-P0)", "subset": _subset_name,
                "reranker_family": family, "candidate_pool_depth": int(depth),
                "user_scope": _scope, "regime": _regime, "n_cases": n,
                "n_users": int(_sub["user_id"].nunique()),
                "mean_full_minus_s2p": float(_sub["delta_full_minus_s2p"].mean()) if n else float("nan"),
                "mean_p1only_minus_p0": float(_sub["delta_stage1"].mean()) if n else float("nan"),
                "mean_did_interaction": m, "ci_low": lo, "ci_high": hi,
                "ci_excludes_zero": bool(np.isfinite(lo) and np.isfinite(hi) and (lo > 0 or hi < 0)),
                "interaction_verdict": _verdict(m, lo, hi),
                "bootstrap_unit": "user_id_cluster", "bootstrap_reps": INTERACTION_BOOT_REPS,
                "bootstrap_seed": INTERACTION_SEED, "post_hoc_label": INTERACTION_POST_HOC,
            })
stage_interaction_I_df = pd.DataFrame(_rows)
stage_interaction_I_df.to_csv(OUT_DIR / f"stage_interaction_I_{CATEGORY_ID}.csv", index=False, encoding="utf-8-sig")
print("NEW-I interaction (I) written. headline depth=1000 overall rows:",
      int((stage_interaction_I_df["candidate_pool_depth"].eq(1000) & stage_interaction_I_df["user_scope"].eq("overall")).sum()))


NEW-I interaction (I) written. headline depth=1000 overall rows: 4


## Interaction II: Fitted-Policy 2×2 Interaction

Interaction II is:

\[
(\mathrm{Full-RankP})-(\mathrm{RetrP-Base}).
\]

It crosses candidate source with the presence of prior-aware reranking features while comparing fitted policies within each learned reranker family. The join is therefore performed by case, candidate depth, and reranker family.

The stored execution includes LightGBM, GAM, and Transformer. Shannon Entropy is absent by design because no separate Shannon RetrP fit exists. The 9,840 Full-minus-RankP rows excluded from the common join are therefore Shannon rows, not missing learned-model observations.

This interaction is post-hoc and descriptive. It is reported with a 2,000-replicate case/user bootstrap interval and no randomization p-value. A confidence interval containing zero indicates non-detection of departure from additivity; it does not prove complementarity, additivity, or equivalence.


In [13]:
# ==== Post-hoc Interaction II: (Full - RankP) - (RetrP - Base) ====
required_upstream_objects = [
    'rank_view_df',
    'bottleneck_rank_promotion_by_reranker_df'
]
missing_upstream_objects = [
    name for name in required_upstream_objects if name not in globals()
]
if missing_upstream_objects:
    raise RuntimeError(
        "Run '# Shared Stage 1 exposure transition and reranker promotion' before NEW-II interaction."
        f" Missing: {missing_upstream_objects}"
    )

INTERACTION_II_POST_HOC = "exploratory_secondary_descriptive_no_outcome_based_search"
INTERACTION_II_BOOT_REPS = 2000
INTERACTION_II_SEED = 42
# The active RetrP condition code is b02. The legacy Y10 variable name is
# retained only for compatibility with the stored execution.
Y10_CONDITION_CODE = "b02"

_have_y10 = bool(rank_view_df["stage_condition"].astype(str).eq(Y10_CONDITION_CODE).any())
if not _have_y10:
    stage_interaction_II_df = pd.DataFrame()
    print(
        "NEW-II: RetrP condition absent from canonical (source stage_condition=='%s' not found). "
        "(II) is a FAIL-CLOSED no-op until 11e/12e/13e are executed and folded into NB14. "
        "Cell structure validated; no CSV emitted." % Y10_CONDITION_CODE
    )
else:
    # ---- (10 - Base) per case x depth x family (family-dependent; parity: Base with S1-Q->S1-P pool) ----
    _y10_frames = []
    for _family in RERANKER_FAMILIES:
        _has_fam = bool(
            (rank_view_df["stage_condition"].astype(str).eq(Y10_CONDITION_CODE)
             & rank_view_df["method_family"].eq(_family)).any()
        )
        if _has_fam:
            _y10_frames.append(
                pair_rank_conditions(rank_view_df, _family, "P2-Q", Y10_CONDITION_CODE, "RetrP_vs_Base")
            )
    if not _y10_frames:
        raise RuntimeError("NEW-II: 10 present but no reranker family carries 10 rows.")
    _y10 = pd.concat(_y10_frames, ignore_index=True, sort=False)[
        ["case_id", "candidate_pool_depth", "regime", "reranker_family", "ndcg_at_5_difference"]
    ].rename(columns={"ndcg_at_5_difference": "delta_y10_minus_s2q"}).copy()
    _y10["case_id"] = _y10["case_id"].astype(str)

    # ---- (Full - RankP) per case x depth x family (reuse existing frame) ----
    _fs = bottleneck_rank_promotion_by_reranker_df.loc[
        bottleneck_rank_promotion_by_reranker_df["contrast_name"].eq("Full_vs_P2-P"),
        ["case_id", "candidate_pool_depth", "regime", "reranker_family", "ndcg_at_5_difference"],
    ].rename(columns={"ndcg_at_5_difference": "delta_full_minus_s2p"}).copy()
    _fs["case_id"] = _fs["case_id"].astype(str)

    # ---- common case_id intersection (inner) on case x depth x family ----
    _n_fs, _n_y10 = len(_fs), len(_y10)
    interaction_ii_did = _fs.merge(
        _y10, on=["case_id", "candidate_pool_depth", "reranker_family"],
        how="inner", suffixes=("", "_y10"), validate="one_to_one",
    )
    if "regime_y10" in interaction_ii_did.columns:
        _mis = interaction_ii_did["regime"].astype(str).ne(interaction_ii_did["regime_y10"].astype(str))
        if bool(_mis.any()):
            raise RuntimeError(
                "NEW-II regime mismatch between (Full-RankP) and (RetrP-Base) on %d rows." % int(_mis.sum())
            )
        interaction_ii_did = interaction_ii_did.drop(columns=["regime_y10"])
    interaction_ii_did["did_interaction"] = (
        interaction_ii_did["delta_full_minus_s2p"] - interaction_ii_did["delta_y10_minus_s2q"]
    )
    print(
        f"NEW-II join: (Full-RankP)={_n_fs} rows, (RetrP-Base)={_n_y10} rows -> "
        f"common={len(interaction_ii_did)} (dropped {_n_fs - len(interaction_ii_did)} / "
        f"{_n_y10 - len(interaction_ii_did)})."
    )

    # ---- case -> user / qchs maps ----
    _umap2 = (rank_view_df.loc[rank_view_df["stage_condition"].eq("P0")]
              .drop_duplicates("case_id").set_index("case_id")["user_id"])
    _qmap2 = (rank_view_df.drop_duplicates("case_id").set_index("case_id")["qchs_profile_available"])
    interaction_ii_did["user_id"] = interaction_ii_did["case_id"].map(_umap2)
    interaction_ii_did["qchs_active"] = boolean_series(
        interaction_ii_did["case_id"].map(_qmap2)).fillna(False).astype(bool)

    _boot_rng2 = np.random.default_rng(INTERACTION_II_SEED)

    def _cluster_boot_ci_ii(frame, value_col):
        v = frame[[value_col, "user_id"]].dropna()
        if len(v) < 2:
            return (float("nan"), float("nan"))
        g = v.groupby("user_id")[value_col].agg(["sum", "count"])
        usum = g["sum"].to_numpy(dtype=float); ucnt = g["count"].to_numpy(dtype=float); G = len(usum)
        idx = _boot_rng2.integers(0, G, size=(INTERACTION_II_BOOT_REPS, G))
        boot = usum[idx].sum(axis=1) / ucnt[idx].sum(axis=1)
        lo, hi = np.quantile(boot, [0.025, 0.975])
        return float(lo), float(hi)

    def _verdict_ii(mean_did, lo, hi):
        if not (np.isfinite(lo) and np.isfinite(hi)):
            return "insufficient_n"
        if lo > 0:
            return "synergy_superadditive"
        if hi < 0:
            return "partial_substitution_subadditive"
        return "additive_complementarity_no_significant_synergy"

    _rows = []
    _subsets = [("full", interaction_ii_did),
                ("qchs_active", interaction_ii_did.loc[interaction_ii_did["qchs_active"]])]
    for _subset_name, _dsub in _subsets:
        for (family, depth) in sorted(
            {(f, int(d)) for f, d in _dsub[["reranker_family", "candidate_pool_depth"]].itertuples(index=False)}):
            _grp = _dsub.loc[_dsub["reranker_family"].eq(family) & _dsub["candidate_pool_depth"].eq(depth)]
            _scopes = [("overall", "overall", _grp)] + [
                ("regime", str(r), g) for r, g in sorted(_grp.groupby("regime", observed=True), key=lambda x: str(x[0]))]
            for _scope, _regime, _sub in _scopes:
                d = _sub["did_interaction"].to_numpy(dtype=float)
                n = int(len(d)); m = float(np.mean(d)) if n else float("nan")
                lo, hi = _cluster_boot_ci_ii(_sub, "did_interaction")
                _rows.append({
                    "category_id": CATEGORY_ID, "interaction": "(II) rerank-prior x pool-personalization",
                    "did_formula": "(Full-RankP)-(RetrP-Base)", "subset": _subset_name,
                    "reranker_family": family, "candidate_pool_depth": int(depth),
                    "user_scope": _scope, "regime": _regime, "n_cases": n,
                    "n_users": int(_sub["user_id"].nunique()),
                    "mean_full_minus_s2p": float(_sub["delta_full_minus_s2p"].mean()) if n else float("nan"),
                    "mean_y10_minus_s2q": float(_sub["delta_y10_minus_s2q"].mean()) if n else float("nan"),
                    "mean_did_interaction": m, "ci_low": lo, "ci_high": hi,
                    "ci_excludes_zero": bool(np.isfinite(lo) and np.isfinite(hi) and (lo > 0 or hi < 0)),
                    "interaction_verdict": _verdict_ii(m, lo, hi),
                    "bootstrap_unit": "user_id_cluster", "bootstrap_reps": INTERACTION_II_BOOT_REPS,
                    "bootstrap_seed": INTERACTION_II_SEED, "post_hoc_label": INTERACTION_II_POST_HOC,
                })
    stage_interaction_II_df = pd.DataFrame(_rows)
    stage_interaction_II_df.to_csv(
        OUT_DIR / f"stage_interaction_II_{CATEGORY_ID}.csv", index=False, encoding="utf-8-sig")
    print("NEW-II interaction (II) written. headline depth=1000 overall rows:",
          int((stage_interaction_II_df["candidate_pool_depth"].eq(1000)
               & stage_interaction_II_df["user_scope"].eq("overall")).sum()))


NEW-II join: (Full-RankP)=39360 rows, (RetrP-Base)=29520 rows -> common=29520 (dropped 9840 / 0).
NEW-II interaction (II) written. headline depth=1000 overall rows: 3
